# Pivotowanie w pandas i polars — pivot_table, pivot, melt, stack/unstack, crosstab, pułapki

**Problem:** pivotowanie (przekształcanie danych z formatu długiego na szeroki i z powrotem) ma w obu bibliotekach kilka nakładających się na siebie narzędzi (`pivot`, `pivot_table`, `crosstab`, `stack`/`unstack`), które łatwo pomylić — zwłaszcza że różnią się tym, czy agregują duplikaty, czy się na nich wywalają.

**Porównanie:**
- `pd.pivot()` / `pl.DataFrame.pivot()` bez `aggregate_function` — czysty reshape, **wymaga unikalnych kombinacji** indeksu i kolumny; przy duplikatach rzuca błąd.
- `pd.pivot_table()` / `pl.DataFrame.pivot(..., aggregate_function=...)` — reshape **z agregacją**, radzi sobie z duplikatami.
- `melt()` / `unpivot()` — operacja odwrotna: z formatu szerokiego z powrotem do długiego.

**Kiedy stosować:** `pivot`/`pivot()` bez agregacji, gdy wiesz na pewno, że kombinacje są unikalne (i chcesz, żeby duplikat był zgłoszony jako błąd, nie po cichu zsumowany); `pivot_table`/`pivot(aggregate_function=...)`, gdy dane mają wiele wierszy na tę samą komórkę wynikowego pivota.

## Setup

Region `Central` ma dane tylko za styczeń — celowo, żeby zademonstrować obsługę brakujących kombinacji w dalszych sekcjach.

In [ ]:
import pandas as pd
import polars as pl

data = {
    "region": ["North", "North", "North", "South", "South", "South",
               "East", "East", "West", "West", "Central"],
    "month": ["Jan", "Jan", "Feb", "Jan", "Jan", "Feb",
              "Jan", "Feb", "Jan", "Feb", "Jan"],
    "category": ["A", "B", "A", "A", "B", "A", "A", "B", "B", "A", "A"],
    "sales": [1200, 300, 1100, 900, 1500, 1000, 700, 950, 1300, 800, 600],
    "units": [12, 3, 11, 9, 15, 10, 7, 10, 13, 8, 6],
}

df = pd.DataFrame(data)
df_pl = pl.DataFrame(data)

df

## Sekcja 1 — Podstawy: `pivot_table()` (pandas) / `pivot()` (polars)

Dane mają dla `(region, month)` po kilka wierszy (różne `category`), więc potrzebna jest agregacja — `aggfunc`/`aggregate_function` mówi, jak połączyć wiele wartości w jedną komórkę.

In [ ]:
# pandas
pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum")

In [ ]:
# polars
df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum")

## Sekcja 2 — Braki po pivotowaniu: `fill_value` (pandas) vs `null`/`0` (polars)

`Central` nie ma danych za luty — po pivotowaniu ta komórka jest pusta. **Sposób, w jaki każda biblioteka to "puste" reprezentuje, różni się w zależności od funkcji agregującej** — to realna pułapka, patrz Sekcja 8.

In [ ]:
# pandas - domyślnie NaN, jawny fill_value zamienia na wybraną wartość
print(pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum"))
print()
print(pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum", fill_value=0))

In [ ]:
# polars - .fill_null() po pivotowaniu, jeśli w ogóle potrzebne (patrz pułapka w Sekcji 8)
df_pl.pivot(on="month", index="region", values="sales", aggregate_function="mean").fill_null(0)

## Sekcja 3 — Wiele kolumn wartości naraz

pandas tworzy w wyniku dwupoziomowy `MultiIndex` kolumn (`(sales, Jan)`, `(sales, Feb)`, `(units, Jan)`...) — trzeba go spłaszczyć ręcznie przed dalszym użyciem (np. eksportem do CSV). polars robi to automatycznie, generując płaskie nazwy typu `sales_Jan`.

In [ ]:
# pandas - MultiIndex kolumn
pivoted = pd.pivot_table(
    df, index="region", columns="month", values=["sales", "units"], aggfunc="sum", fill_value=0
)
print(pivoted.columns.tolist())
pivoted

In [ ]:
# Spłaszczenie MultiIndex kolumn do pojedynczego poziomu z czytelnymi nazwami
pivoted.columns = [f"{value}_{month}" for value, month in pivoted.columns]
pivoted

In [ ]:
# polars - płaskie nazwy kolumn od razu, bez dodatkowego kroku
df_pl.pivot(on="month", index="region", values=["sales", "units"], aggregate_function="sum")

### Różna agregacja dla różnych kolumn wartości: `aggfunc` jako słownik

Czasem `sales` ma sens sumować, a `units` uśredniać (np. średnia wielkość zamówienia) — w jednym wywołaniu `pivot_table`, przez podanie `aggfunc` jako słownika `{kolumna: funkcja}`.

In [ ]:
pd.pivot_table(
    df, index="region", columns="month",
    values=["sales", "units"],
    aggfunc={"sales": "sum", "units": "mean"},
    fill_value=0,
)

### Kilka agregacji dla TEJ SAMEJ kolumny wartości: `aggfunc` jako lista

Gdy potrzebujesz jednocześnie sumy, średniej i liczby transakcji z tej samej kolumny — lista funkcji zamiast pojedynczej nazwy, bez wielokrotnego wywoływania `pivot_table`.

In [ ]:
pd.pivot_table(
    df, index="region", columns="month",
    values="sales", aggfunc=["sum", "mean", "count"], fill_value=0,
)

### Wiele kolumn `index` naraz: zagnieżdżone wiersze

`index=["region", "category"]` tworzy hierarchiczny (`MultiIndex`) układ wierszy — dwa poziomy grupowania po lewej stronie tabeli, zamiast jednego. Naturalne rozszerzenie tego, co już widziałeś przy `columns=` — tu ta sama logika działa dla wierszy.

In [ ]:
pd.pivot_table(
    df, index=["region", "category"], columns="month",
    values="sales", aggfunc="sum", fill_value=0,
)

### To samo w polars: wiele kolumn `index` naraz

polars przyjmuje listę w `index=` dokładnie tak samo — różnica jest tylko wizualna: zamiast hierarchicznego `MultiIndex` po lewej stronie, `region` i `category` zostają zwykłymi, płaskimi kolumnami (bo polars w ogóle nie ma pojęcia indeksu).

In [ ]:
df_pl.pivot(on="month", index=["region", "category"], values="sales", aggregate_function="sum")

## Sekcja 4 — Sumy brzegowe (`margins`)

pandas ma wbudowany parametr `margins=True`, który dokłada wiersz i kolumnę z sumą całkowitą. **polars nie ma odpowiednika** — trzeba dodać wiersz/kolumnę sumaryczną ręcznie.

In [ ]:
# pandas
pd.pivot_table(
    df, index="region", columns="month", values="sales", aggfunc="sum",
    fill_value=0, margins=True, margins_name="Razem",
)

In [ ]:
# polars - ręczne dodanie kolumny i wiersza sumy
result = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum").fill_null(0)

month_cols = [c for c in result.columns if c != "region"]
result = result.with_columns(pl.sum_horizontal(month_cols).alias("Razem"))

total_row = result.select([pl.lit("Razem").alias("region")] + [pl.col(c).sum() for c in month_cols + ["Razem"]])
pl.concat([result, total_row])

## Sekcja 5 — `pivot()` bez agregacji: różnica względem `pivot_table()`

`pivot()`/`pivot(aggregate_function=None)` **zakłada unikalne kombinacje** indeksu i kolumny — nic nie agreguje. To zaleta, jeśli chcesz, żeby duplikat był głośnym błędem, a nie cichą sumą (patrz też Sekcja 8).

In [ ]:
# pandas - unikalna kombinacja (region, category) -> działa bez agregacji
df.pivot(index=["region", "category"], columns="month", values="sales")

## Sekcja 6 — `melt()` / `unpivot()`: odwrotność pivota

Format szeroki z powrotem do długiego — przydatne np. przed wykresem w seaborn/plotly, który oczekuje jednej kolumny z wartością i jednej z kategorią.

In [ ]:
wide = pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum", fill_value=0).reset_index()
print(wide)
print()

# pandas
wide.melt(id_vars="region", var_name="month", value_name="sales")

In [ ]:
# polars - nazywa się unpivot (dawniej też melt, przestarzałe)
wide_pl = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum").fill_null(0)
wide_pl.unpivot(index="region", variable_name="month", value_name="sales")

## Sekcja 7 — `stack()` / `unstack()` (pandas, dane z `MultiIndex`)

Gdy dane już mają hierarchiczny indeks (np. wynik `groupby()` po dwóch kolumnach), `unstack()` przenosi jeden poziom indeksu do kolumn — de facto alternatywa dla `pivot_table` na danych już zgrupowanych. `stack()` robi odwrotność.

In [ ]:
grouped = df.groupby(["region", "month"])["sales"].sum()
print(grouped)
print()

unstacked = grouped.unstack()
print(unstacked)
print()

unstacked.stack()

## Sekcja 8 — `crosstab()` (pandas): skrót do tabeli częstości

Gdy interesuje Cię tylko **liczba wystąpień** kombinacji (nie suma jakiejś wartości), `crosstab()` jest krótsze niż `pivot_table(..., aggfunc="count")`. Przyjmuje też `values=`/`aggfunc=`, jeśli jednak potrzebna agregacja innej kolumny.

In [ ]:
# Liczba transakcji w każdej kombinacji region x month
print(pd.crosstab(df["region"], df["month"]))
print()

# To samo co pivot_table z aggfunc='sum', ale krócej
pd.crosstab(df["region"], df["month"], values=df["sales"], aggfunc="sum")

### `crosstab` z `normalize`: proporcje zamiast liczby wystąpień

Zamiast surowych liczb, `normalize=` przelicza tabelę na proporcje — trzy różne "punkty odniesienia" do wyboru, w zależności od pytania, na które odpowiadasz:
- `normalize="index"` — proporcje W RAMACH każdego wiersza (sumują się do 1 w poziomie). Pytanie: "jaki % sprzedaży regionu N przypada na kategorię A?"
- `normalize="columns"` — proporcje W RAMACH każdej kolumny (sumują się do 1 w pionie). Pytanie: "jaki % całej kategorii A pochodzi z regionu N?"
- `normalize="all"` — proporcje z CAŁEJ tabeli naraz (wszystkie komórki sumują się do 1). Pytanie: "jaki % wszystkich transakcji to akurat region N + kategoria A?"

In [ ]:
print("normalize='index' - proporcje w ramach WIERSZA:")
print(pd.crosstab(df["region"], df["category"], normalize="index").round(2))
print()
print("normalize='columns' - proporcje w ramach KOLUMNY:")
print(pd.crosstab(df["region"], df["category"], normalize="columns").round(2))
print()
print("normalize='all' - proporcje z CAŁEJ tabeli:")
print(pd.crosstab(df["region"], df["category"], normalize="all").round(2))

### `crosstab` z `margins`: sumy brzegowe też dla tabeli częstości

Ten sam parametr co w `pivot_table` (Sekcja 4) — dokłada wiersz i kolumnę "Razem".

In [ ]:
pd.crosstab(df["region"], df["category"], margins=True, margins_name="Razem")

### `unstack` z `fill_value`: to samo co `fill_value` w `pivot_table`, tylko na już zgrupowanych danych

Domyślnie brakujące kombinacje po `unstack()` to `NaN` — dokładnie tak samo jak przy `pivot_table` bez `fill_value` (Sekcja 2).

In [ ]:
grouped = df.groupby(["region", "month"])["sales"].sum()

print("Bez fill_value:")
print(grouped.unstack())
print()
print("Z fill_value=0:")
print(grouped.unstack(fill_value=0))

### polars: odpowiednik `crosstab` — `pivot` z `aggregate_function="len"`

polars nie ma osobnej funkcji `crosstab` — ten sam efekt (tabela częstości) osiąga się przez zwykły `pivot`, licząc długość grupy zamiast sumy wartości.

In [ ]:
df_pl.pivot(
    on="category", index="region", values="category", aggregate_function="len"
).fill_null(0)

## Sekcja 9 — Pułapki

### Pułapka 1 — `pivot()` bez agregacji rzuca błąd przy duplikatach (w obu bibliotekach)

To zachowanie jest właściwie zaletą — `pivot()` **nie zgadnie za Ciebie**, jak połączyć duplikaty, tylko głośno się zatrzyma. Jeśli oczekujesz błędu, a dostajesz wynik bez niego, to sygnał, że przypadkiem użyłeś `pivot_table()`/`aggregate_function` zamiast czystego `pivot()`.

In [ ]:
# pandas: (region, month) ma duplikaty (różne category) -> błąd
try:
    df.pivot(index="region", columns="month", values="sales")
except ValueError as e:
    print(f"pandas pivot() - błąd: {e}")

print()

# polars: to samo bez aggregate_function
try:
    df_pl.pivot(on="month", index="region", values="sales")
except Exception as e:
    print(f"polars pivot() bez aggregate_function - błąd: {type(e).__name__}")

### Pułapka 2 — polars: `sum` traktuje brak danych jako `0`, `mean` jako `null`

To rozbieżność, która łatwo umyka: przy `aggregate_function="sum"` brakująca kombinacja (`Central`/`Feb`) dostaje **`0`**, a nie `null` — bo suma pustego zbioru to matematycznie zero. Przy `aggregate_function="mean"` ta sama brakująca kombinacja to **`null`** — bo średniej z pustego zbioru nie da się policzyć. Efekt: `0` po `sum` wygląda identycznie jak prawdziwa, zarejestrowana sprzedaż zerowa — nie da się ich odróżnić bez sprawdzenia danych źródłowych.

In [ ]:
sum_pivot = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum")
mean_pivot = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="mean")

print("aggregate_function='sum' - Central/Feb:")
print(sum_pivot.filter(pl.col("region") == "Central"))

print("\naggregate_function='mean' - Central/Feb:")
print(mean_pivot.filter(pl.col("region") == "Central"))

### Pułapka 3 — `fill_value`/`fill_null` nie odróżnia "braku danych" od "wartości NaN w źródle"

Jeśli w danych źródłowych `sales` bywa `NaN` (np. niekompletny wpis, nie brak kombinacji), po `fill_value=0` obie sytuacje — "nie było takiego wiersza" i "był wiersz, ale z brakującą wartością" — wyglądają identycznie w wyniku. Zanim zastosujesz `fill_value`, warto sprawdzić `df["sales"].isna().sum()` na danych źródłowych, żeby wiedzieć, czy to rozróżnienie ma znaczenie w Twoim przypadku.

## Podsumowanie

| Zadanie | pandas | polars |
|---|---|---|
| Pivot z agregacją (duplikaty OK) | `pd.pivot_table(df, index=, columns=, values=, aggfunc=)` | `df.pivot(on=, index=, values=, aggregate_function=)` |
| Pivot bez agregacji (duplikaty = błąd) | `df.pivot(index=, columns=, values=)` | `df.pivot(on=, index=, values=)` (bez `aggregate_function`) |
| Wypełnienie braków po pivotowaniu | `fill_value=0` w `pivot_table` | `.fill_null(0)` po `pivot` |
| Wiele kolumn wartości naraz | `values=["a","b"]` → wymaga spłaszczenia `MultiIndex` kolumn | `values=["a","b"]` → płaskie nazwy od razu |
| Różna agregacja dla różnych kolumn wartości | `aggfunc={"a":"sum","b":"mean"}` | osobne wywołania `pivot` lub `group_by().agg()` |
| Kilka agregacji dla tej samej kolumny wartości | `aggfunc=["sum","mean","count"]` | osobne wywołania `pivot` lub `group_by().agg()` |
| Wiele kolumn `index` naraz (zagnieżdżone wiersze) | `index=["a","b"]` → hierarchiczny `MultiIndex` | `index=["a","b"]` → zwykłe, płaskie kolumny |
| Sumy brzegowe (wiersz/kolumna "Razem") | `margins=True` | brak wbudowanego — dodaj ręcznie (`sum_horizontal` + wiersz sumy) |
| Format szeroki → długi | `df.melt(id_vars=, var_name=, value_name=)` | `df.unpivot(index=, variable_name=, value_name=)` |
| Reshape na już zgrupowanych danych z `MultiIndex` | `.unstack(fill_value=)` / `.stack()` | nie dotyczy (polars nie ma indeksu) |
| Tabela częstości (liczba wystąpień) | `pd.crosstab(a, b)` | `df.pivot(..., aggregate_function="len")` |
| Tabela częstości jako proporcje, nie liczby | `pd.crosstab(a, b, normalize="index"/"columns"/"all")` | brak wprost — podziel ręcznie przez sumę |
| Sumy brzegowe w tabeli częstości | `pd.crosstab(a, b, margins=True)` | brak wprost |

**Wniosek:** najbezpieczniejszy nawyk to domyślnie sięgać po `pivot()` (bez agregacji) tam, gdzie kombinacje *powinny* być unikalne — błąd przy duplikacie wychwyci problem jakości danych, zanim zostanie po cichu zsumowany przez `pivot_table()`. `pivot_table()`/`aggregate_function` zostaw świadomie dla przypadków, gdzie agregacja jest zamierzona.